In [0]:
import usaddress
from typing import Any

# usaddress tags that together form the street line, in output order.
STREET_PARTS: list[str] = [
    "AddressNumberPrefix",
    "AddressNumber",
    "AddressNumberSuffix",
    "StreetNamePreDirectional",
    "StreetNamePreModifier",
    "StreetNamePreType",
    "StreetName",
    "StreetNamePostType",
    "StreetNamePostDirectional",
    "StreetNamePostModifier",
]

# Output contract for parsed address fields.
PARSED_COLUMNS: list[str] = ["Address_1", "Address_2", "City", "State", "Zip_Code", "Parsing_Error"]

_EMPTY_PARSE: dict[str, None] = {c: None for c in PARSED_COLUMNS}


def parse_address(address: str | None) -> dict[str, str | None]:
    """Parse a raw US address string into standard components.

    Args:
        address: Raw address string to parse.

    Returns:
        Dict containing Address_1, Address_2, City, State, Zip_Code, and Parsing_Error.
    """
    if not address:
        return {**_EMPTY_PARSE, "Parsing_Error": "Empty input"}

    try:
        tagged, _ = usaddress.tag(address)
    except usaddress.RepeatedLabelError:
        return {**_EMPTY_PARSE, "Parsing_Error": "RepeatedLabelError"}
    except Exception as e:
        return {**_EMPTY_PARSE, "Parsing_Error": str(e)}

    street = " ".join(p for p in (tagged.get(k) for k in STREET_PARTS) if p)
    suite_type = tagged.get("OccupancyType")
    suite_number = tagged.get("OccupancyIdentifier")

    return {
        "Address_1": street or None,
        "Address_2": " ".join(p for p in (suite_type, suite_number) if p) if suite_number else None,
        "City": tagged.get("PlaceName"),
        "State": tagged.get("StateName"),
        "Zip_Code": tagged.get("ZipCode"),
        "Parsing_Error": None,
    }